# Search users

## What it does

Shows the users whose name contains a piece of text, ordered by name. Spaces around the text do not matter, and neither do capitals for the letters A to Z.

## What happens to the saved users

It creates three practice users, searches them, and deletes them at the end. The real users stay the same.

## Feature it uses

`searchUsers(text)`, from `logic/search-users.js`.

## Failures it can show

- A text no name contains → only the header.
- An empty text → `Name is empty`.
- The users cannot be read → `Could not find the user`.

Open it with `pnpm jupyter` from `first-app`.

In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent

code = r"""
try {
  const { default: Table } = await import("cli-table3");
  const { createUser } = await import("./logic/create-user.js");
  const { searchUsers } = await import("./logic/search-users.js");
  const { deleteUser } = await import("./logic/delete-user.js");

  function printUsers(users) {
    const table = new Table({
      head: ["id", "name", "email", "createdAt", "updatedAt"],
      style: { head: ["cyan", "bold"] },
    });
    for (const user of users) {
      table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
    }
    console.log(table.toString());
  }

  const mark = `Zq${Date.now()}`;
  const adam = createUser(`${mark} Adam`, `practice-${Date.now()}-1@x.com`);
  try {
    const ada = createUser(`${mark} Ada`, `practice-${Date.now()}-2@x.com`);
    try {
      const maria = createUser(`${mark} María`, `practice-${Date.now()}-3@x.com`);
      try {
        // 1. A piece in other capitals, with spaces around: a table with Ada then Adam, not María
        printUsers(searchUsers(`  ${mark.toLowerCase()} AD  `));

        // 2. A percent sign is a plain character: only the header
        printUsers(searchUsers("%"));

        // 3. Only spaces: "Name is empty"
        try {
          searchUsers("   ");
        } catch (error) {
          console.log(error.message);
        }
      } finally {
        deleteUser(maria.id);
      }
    } finally {
      deleteUser(ada.id);
    }
  } finally {
    deleteUser(adam.id);
  }
} catch (error) {
  console.log(error.message);
}
"""

if (app / "logic").is_dir():
    env = os.environ.copy()
    env["FORCE_COLOR"] = "1"
    env.pop("NO_COLOR", None)
    completed = subprocess.run(
        ["node", "--no-warnings", "--input-type=module", "-e", code],
        cwd=app,
        capture_output=True,
        text=True,
        env=env,
    )
    print(completed.stdout, end="")
    if completed.returncode != 0:
        print(completed.stderr, end="")
else:
    print("Cannot find first-app.")

┌────┬──────────────────────┬────────────────────────────────┬─────────────────────┬─────────────────────┐
│ id │ name                 │ email                          │ createdAt           │ updatedAt           │
├────┼──────────────────────┼────────────────────────────────┼─────────────────────┼─────────────────────┤
│ 2  │ Zq1793720863661 Ada  │ practice-1793720863663-2@x.com │ 2026-11-03 16:47:43 │ 2026-11-03 16:47:43 │
├────┼──────────────────────┼────────────────────────────────┼─────────────────────┼─────────────────────┤
│ 1  │ Zq1793720863661 Adam │ practice-1793720863661-1@x.com │ 2026-11-03 16:47:43 │ 2026-11-03 16:47:43 │
└────┴──────────────────────┴────────────────────────────────┴─────────────────────┴─────────────────────┘
┌────┬──────┬───────┬───────────┬───────────┐
│ id │ name │ email │ createdAt │ updatedAt │
└────┴──────┴───────┴───────────┴───────────┘
Name is empty
